<a href="https://colab.research.google.com/github/John-Fernand/Projeto_Integrador_Faculdade/blob/main/SteamDatabase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U "kagglehub>=1.0.3" "certifi>=2026.10.19"
!apt-get install -y fonts-noto-cjk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  fonts-noto-cjk-extra
The following NEW packages will be installed:
  fonts-noto-cjk
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 61.2 MB of archives.
After this operation, 93.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 fonts-noto-cjk all 1:20230817+repack1-3 [61.2 MB]
Fetched 61.2 MB in 4s (16.4 MB/s)
Selecting previously unselected package fonts-noto-cjk.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../fonts-noto-cjk_1%3a20230817+repack1-3_all.deb ...
Unpacking fonts-noto-cjk (1:20230817+repack1-3) ...
Setting up fonts-noto-cjk (1:20230817+repack1-3) ...
Processing triggers for fontconfig (2.15.0-1.1ubuntu2) ...


In [ ]:
import pandas as pd
from google.colab import userdata
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import time
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

fonte = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'

fm.fontManager.addfont(fonte)
plt.rcParams['font.family'] = 'Noto Sans CJK JP'

pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', 5)

pd.set_option('display.width', None)

pd.set_option('display.max_colwidth', None)

In [ ]:
class SteamVendas:

    def carregar_dados(self):
        token = userdata.get("KAGGLE_API_TOKEN")

        if token:
            os.environ["KAGGLE_API_TOKEN"] = token
            print("Autenticação OK!")
        else:
            print("Token não encontrado.")
            return False

        path = kagglehub.dataset_download(
            "fronkongames/steam-games-dataset"
        )

        print("Dataset baixado em:", path)

        for root, dirs, files in os.walk(path):
            for file in files:
                print(file)

        file_path = os.path.join(path, "games.csv")

        self.df = pd.read_csv(
            file_path,
            index_col=False
        ).reset_index(drop=True)

        print("Dataset carregado com sucesso!")

        self.Treinar_Recomendador()

        return True

    def menu(self):
        print("\n===== Escolhas =====")
        print("1. Procurar jogos.")
        print("2. Adicionar jogos.")
        print("3. Limpar a lista.")
        print("4. Exibir gráfico de jogos mais populares.")
        print("5. Jogos recomendados.")
        print("6. Encerrar o processo.")

    def executar(self):

        listajogos = []

        if not self.carregar_dados():
            return

        continuar = True

        while continuar:

            self.menu()
            escolha = input("Digite a opção desejada: ")

            match escolha:

                case "1":
                    self.Procurar_Jogos()

                case "2":
                    self.Adicionar_Jogos(listajogos)

                case "3":
                    self.Limpar_Lista(listajogos)
                    print("Lista limpa com sucesso!")

                case "4":
                    self.Grafico_Jogos_Populares()

                case "5":
                    self.Jogos_Recomendados(listajogos)

                case "6":
                    print("Encerrando o programa...")
                    continuar = False

                case _:
                    print("Opção inválida. Tente novamente.")

    def Procurar_Jogos(self):

        continuar2 = True

        while continuar2:

            try:
                idjogo = int(input("Selecione um jogo pela ID:\n"))

            except ValueError:
                print("Digite uma ID válida.")
                continue

            resultado = self.df[self.df['AppID'] == idjogo]

            if resultado.empty:
                print("Jogo não encontrado.")
            else:
                print(resultado)

            escolhacontinuar2 = input(
                'Deseja continuar?\nDigite exatamente "Não" para parar.\n'
            )

            if escolhacontinuar2 == "Não":
                continuar2 = False

            time.sleep(1)

    def Adicionar_Jogos(self, listajogos):

        continuar2 = True

        while continuar2:

            try:
                idjogo = int(
                    input("Selecione um jogo pela ID para adicionar:\n")
                )

            except ValueError:
                print("Digite uma ID válida.")
                continue

            resultado = self.df[self.df['AppID'] == idjogo]

            if resultado.empty:
                print("Jogo não encontrado.")

            else:
                if any(
                    jogo['AppID'].iloc[0] == idjogo
                    for jogo in listajogos
                ):
                    print("Esse jogo já está na lista.")

                else:
                    print(
                        f"Jogo encontrado: {resultado['Name'].iloc[0]}"
                    )
                    listajogos.append(resultado.copy())

            if listajogos:
                print("\n===== Sua lista de jogos =====")
                display(pd.concat(listajogos)['Name'])

            escolhacontinuar2 = input(
                'Deseja continuar?\nDigite exatamente "Não" para parar.\n'
            )

            if escolhacontinuar2 == "Não":
                continuar2 = False

            time.sleep(1)

    def Limpar_Lista(self, listajogos):

        listajogos.clear()

    def Grafico_Jogos_Populares(self):

        try:
            númeroescolhido = int(
                input("Escolha o número de jogos mostrados: ")
            )

        except ValueError:
            print("Digite um número inteiro.")
            return

        if númeroescolhido <= 0:
            print("Digite um número maior que zero.")
            return

        mais_populares = self.df.nlargest(
            númeroescolhido, 'Peak CCU'
        )

        plt.figure(
            figsize=(12, max(6, len(mais_populares) * 0.35))
        )

        barras = plt.barh(
            mais_populares['Name'],
            mais_populares['Peak CCU']
        )

        plt.title(
            f'Top {len(mais_populares)} jogos mais populares da Steam'
        )
        plt.xlabel('Pico de jogadores simultâneos')
        plt.ylabel('Nome dos jogos')

        plt.bar_label(barras, fmt='{:,.0f}', padding=4)

        plt.gca().invert_yaxis()
        plt.tight_layout()

        plt.show()

    def Treinar_Recomendador(self):

        colunas = [
            'Genres',
            'Categories',
            'Tags'
        ]

        colunas = [
            coluna for coluna in colunas
            if coluna in self.df.columns
        ]

        if not colunas:
            raise ValueError(
                "Nenhuma coluna textual encontrada."
            )

        dados = self.df[colunas].fillna('').astype(str)

        textos = dados.agg(' '.join, axis=1)

        self.vetorizador = TfidfVectorizer(
            stop_words='english',
            max_features=200000,
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.80,
            sublinear_tf=True,
            norm='l2'
        )

        matriz = self.vetorizador.fit_transform(textos)

        self.modelo = NearestNeighbors(
            metric='cosine',
            algorithm='brute'
        )

        self.modelo.fit(matriz)

        self.matriz_jogos = matriz

        print("Recomendador treinado com sucesso!")

    def Jogos_Recomendados(self, listajogos, quantidade=10):

        if not listajogos:
            print("Adicione jogos à lista antes de recomendar.")
            return

        if quantidade <= 0:
            print("A quantidade deve ser maior que zero.")
            return

        ids_selecionados = list(dict.fromkeys(
            jogo['AppID'].iloc[0]
            for jogo in listajogos
        ))

        print("\n===== Jogos recomendados =====")

        for idjogo in ids_selecionados:

            resultado = self.df[
                self.df['AppID'] == idjogo
            ]

            if resultado.empty:
                continue

            indice = resultado.index[0]

            vetor = self.matriz_jogos[indice]

            n_vizinhos = min(
                quantidade + len(ids_selecionados) + 1,
                len(self.df)
            )

            distancias, indices = self.modelo.kneighbors(
                vetor,
                n_neighbors=n_vizinhos
            )

            print(
                f"\n--- Recomendações para: "
                f"{resultado['Name'].iloc[0]} ---"
            )

            recomendacoes = 0

            for distancia, indice_jogo in zip(
                distancias[0],
                indices[0]
            ):

                jogo = self.df.iloc[indice_jogo]

                if jogo['AppID'] in ids_selecionados:
                    continue

                similaridade = (1 - distancia) * 100

                print(
                    f"\nJogo: {jogo['Name']}"
                    f"\nAppID: {jogo['AppID']}"
                    f"\nSimilaridade: {similaridade:.2f}%"
                )

                recomendacoes += 1

                if recomendacoes >= quantidade:
                    break

            if recomendacoes == 0:
                print("Nenhuma recomendação encontrada.")




In [ ]:
programa = SteamVendas()
programa.executar()

Autenticação OK!


100%|██████████| 315M/315M [00:01<00:00, 175MB/s]

Extracting files...


Dataset baixado em: /root/.cache/kagglehub/datasets/fronkongames/steam-games-dataset/versions/60
games.csv
games.json
Dataset carregado com sucesso!
Recomendador treinado com sucesso!

===== Escolhas =====
1. Procurar jogos.
2. Adicionar jogos.
3. Limpar a lista.
4. Exibir gráfico de jogos mais populares.
5. Jogos recomendados.
6. Encerrar o processo.
Digite a opção desejada: 2
Selecione um jogo pela ID para adicionar:
601150
Jogo encontrado: Devil May Cry 5

===== Sua lista de jogos =====


,Name
59927,Devil May Cry 5


Deseja continuar?
Digite exatamente "Não" para parar.

Selecione um jogo pela ID para adicionar:
235460
Jogo encontrado: METAL GEAR RISING: REVENGEANCE

===== Sua lista de jogos =====


,Name
59927,Devil May Cry 5
21543,METAL GEAR RISING: REVENGEANCE


Deseja continuar?
Digite exatamente "Não" para parar.
Não

===== Escolhas =====
1. Procurar jogos.
2. Adicionar jogos.
3. Limpar a lista.
4. Exibir gráfico de jogos mais populares.
5. Jogos recomendados.
6. Encerrar o processo.
Digite a opção desejada: 5

===== Jogos recomendados =====

--- Recomendações para: Devil May Cry 5 ---

Jogo: Resident Evil 2
AppID: 883710
Similaridade: 84.43%

Jogo: MONSTER HUNTER RISE
AppID: 1446780
Similaridade: 83.54%

Jogo: Exoprimal
AppID: 1286320
Similaridade: 76.82%

Jogo: Resident Evil Village
AppID: 1196590
Similaridade: 72.71%

Jogo: Resident Evil 4
AppID: 2050650
Similaridade: 71.96%

Jogo: Resident Evil 7 Biohazard
AppID: 418370
Similaridade: 71.96%

Jogo: Dead Rising Deluxe Remaster
AppID: 2527390
Similaridade: 71.96%

Jogo: Remember Me
AppID: 228300
Similaridade: 69.95%

Jogo: Resident Evil 3
AppID: 952060
Similaridade: 69.63%

Jogo: Dead Rising 4
AppID: 543460
Similaridade: 67.89%

--- Recomendações para: METAL GEAR RISING: REVENGEANCE ---

Jo